入力画像のシイタケを識別し，それぞれに3種類で色付

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO

def main():
    # パスの指定
    model_path = '/home/hiromu/energy/YOLO/0708_maesyori/datasets/train/weights/best.pt'
    image_path = '/home/hiromu/energy/data/1201_humomentstest/org/A/IMG_2642.jpg'
    output_path = 'output_segmented.jpg' # 出力画像のファイル名

    # モデルの読み込み
    print("モデルを読み込んでいます...")
    model = YOLO(model_path)

    # 画像の読み込み
    img = cv2.imread(image_path)
    if img is None:
        print("エラー: 画像を読み込めませんでした。パスが正しいか確認してください。")
        return

    # 推論の実行
    print("画像から対象をセグメンテーションしています...")
    results = model(img)

    # 3色の定義 (OpenCVはBGR形式: 青, 緑, 赤)
    colors = [
        (255, 0, 0),   # 青
        (0, 255, 0),   # 緑
        (0, 0, 255)    # 赤
    ]

    # セグメンテーションマスクの取得
    if results[0].masks is None:
        print("エラー: マスクデータが見つかりませんでした。")
        print("※使用している best.pt がセグメンテーションタスク用（YOLOv8-segなど）で学習されているか確認してください。")
        return

    # 対象の輪郭（ポリゴン座標）リストを取得
    segments = results[0].masks.xy
    
    if len(segments) == 0:
        print("対象が検出されませんでした。")
        return

    # 半透明の塗りつぶし用オーバーレイを作成
    overlay = img.copy()

    # 検出された対象に順番に色を付ける
    for i, segment in enumerate(segments):
        if len(segment) == 0:
            continue
            
        # OpenCVで描画するために、座標を整数のNumPy配列に変換し、形状を調整
        pts = np.array(segment, np.int32)
        pts = pts.reshape((-1, 1, 2))
        
        # 検出順に3色をループで割り当てる
        color = colors[i % 3]
        
        # オーバーレイ画像にポリゴン（対象の形）を塗りつぶし
        cv2.fillPoly(overlay, [pts], color)
        
        # 境界をはっきりさせるために、元画像に同じ色で輪郭線を描画
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=2)

    # 元画像にオーバーレイを合成 (0.5は塗りつぶしの不透明度)
    cv2.addWeighted(overlay, 0.5, img, 0.5, 0, img)

    # 画像の保存
    cv2.imwrite(output_path, img)
    print(f"処理完了: {len(segments)}個の対象をセグメンテーションし、{output_path} に保存しました。")

if __name__ == "__main__":
    main()

モデルを読み込んでいます...
画像から対象をセグメンテーションしています...

0: 640x480 53 shiitakes, 10.9ms
Speed: 1.5ms preprocess, 10.9ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 480)
処理完了: 53個の対象をセグメンテーションし、output_segmented.jpg に保存しました。
